# 11 · RAG：让 Claude 带着出处回答蛋白问题

## 这一章在整个项目里的位置

```
00–10 ✅ → [11 RAG] → 12 Agent → 13 收尾
            知识库 + Claude：问"TP53 是做什么的"，回答里每句话都标着出处   ← 你在这里
```

## 背景：为什么需要这一章

10 号之后，我们的平台能回答"TP53 R175H 的**分数**是多少"。但用的人接着会问：
- TP53 是做什么的？R175H 以前有人报道过吗？和什么病有关？
- QAFI 是怎么验证的？

直接问一个大语言模型（比如 Claude）行不行？有三个问题：
1. **它的知识停在训练那天**，而且不知道你的未发表论文
2. **它会编**：说得很流利，但可能是错的，这叫"幻觉"
3. **没有出处**：你没法核对它说的哪句话来自哪里

**RAG**（Retrieval-Augmented Generation，检索增强生成）的办法是：**先查资料，再回答**。
- 先从我们自己的资料库里找出和问题最相关的几段话
- 再让 Claude **只根据这几段话**回答，每句话标明出自哪一段
- 资料里没有，就老实说不知道

这是现在公司里用大模型最常见的方式：客服机器人、内部文档问答、法规查询，几乎都是 RAG。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2、3 步：UniProt 注释进 raw，整理成 2 万篇文档（约 2 分钟）
   ├─ 第 4、5 步：Terraform 建知识库；上传，导入（约 100 分钟）
   └─ 第 6–8 步：检索、回答、评估
看结果：AWS 控制台（右上角切到 Europe (Ireland)）→ Amazon Bedrock → Knowledge Bases → qafi-dev-knowledge
```

## 这一章做的 8 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | RAG 是怎么工作的 | 开卷考试 |
| 2 | 资料进货 | 把教科书和你的论文搬进图书馆 |
| 3 | 整理成文档 | 每个蛋白一张卡片 |
| 4 | 建知识库 | 按"意思"排架的图书馆 |
| 5 | 上传和导入 | 把卡片切成小段、编好索引、上架 |
| 6 | 检索 | 按意思找到最相关的几段 |
| 7 | 带出处地回答 | 开卷答题，每句注明出处 |
| 8 | 评估 | 出一套有标准答案的题，打分 |

**花费**：一次性约 0.3 美元（把 2 万篇文档变成向量）；之后每个问题约 0.3 美分。**时间**：约 120 分钟，大部分是等导入。

**开始前**：
1. 终端运行 `aws login --profile qafi`
2. **开通 Claude**（每个 AWS 账号做一次）：AWS 控制台，右上角切到 **Europe (Ireland)** → Amazon Bedrock → Model catalog → **Claude Haiku 4.5**：先按提示提交用例说明（个人学习项目，用公开的蛋白注释做问答），再接受按量付费的使用协议（没有月费，费用从免费额度里扣）。第 7 步之前要完成

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import json, os, subprocess, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import pandas as pd
from qafi_platform import claude, ingest, knowledge
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"
pd.set_option("display.max_colwidth", 120)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
def terraform_output(name):
    return json.loads(subprocess.run(f"terraform -chdir=infra/dev output -json {name}", shell=True, capture_output=True, text=True).stdout)
frankfurt, ireland = boto3.Session(region_name="eu-central-1"), boto3.Session(region_name=claude.REGION)
RAW = terraform_output("bucket_names")["raw"]
UNIPROT = "2026_03"
PAPER = Path("../QAFI_CODE_NEW/reports/paper/ijms submission/manuscript.v9.docx")
print("AWS:", frankfurt.client("sts").get_caller_identity()["Arn"], "| 知识库区域:", claude.REGION)

---

## 第 1 步 · RAG 是怎么工作的

### 在干什么
弄懂 RAG 的三个零件：**向量**、**检索**、**生成**。

### 背景
比喻：**开卷考试**。闭卷时学生只能凭记忆，记错了也会写得头头是道（幻觉）。开卷时先翻书找到相关的几页，照着书答，还能注明"见第几页"。RAG 就是让大模型开卷考试。

**零件一：向量（embedding）**。怎么"按意思"找资料？用一个**嵌入模型**把一段话变成一串数字（我们用 Titan Text Embeddings V2，每段话变成 1,024 个数）。神奇之处在于：**意思相近的话，数字也相近**。

```
"TP53 是一个抑癌蛋白"            → [0.12, -0.53, 0.88, ...]  ┐ 很接近
"守护基因组、DNA 损伤后让细胞凋亡" → [0.10, -0.49, 0.91, ...]  ┘
"血红蛋白运输氧气"               → [-0.70, 0.33, 0.05, ...]     离得远
```

所以即使问题里没有"TP53"这个词，也能找到 TP53 的那段。这和 08 号的 ESM 向量是同一个思想：ESM 把氨基酸序列变成向量，这里把一段文字变成向量。

**零件二：检索**。资料先切成一小段一小段（每段约 300 个词），每段算好向量，存进**向量数据库**。提问时，把问题也变成向量，找出离它最近的几段。

**零件三：生成**。把找到的几段话编上号，连同问题一起交给 Claude，要求它：**只用这几段**回答，每句话后面标 [1] [2]，找不到就说找不到。

```
问题 ──嵌入──▶ 向量 ──最近的 5 段──▶ [1] [2] [3] [4] [5] ──┐
                         ▲                                  ├──▶ Claude ──▶ 带 [n] 出处的回答
              2 万篇文档切成的几万段（向量数据库）   问题 ───┘
```

**为什么不把资料"训练"进模型（微调）？** 资料会更新（UniProt 每 8 周一版），重新训练很贵；而且训练进去的知识没法注明出处。RAG 更新资料只要重新导入，每句话都能追溯。

### 用了什么技术，为什么
- **Amazon Bedrock Knowledge Bases**：AWS 托管的 RAG 服务。切段、算向量、存索引、检索都由它负责
- **Titan Text Embeddings V2**：AWS 自己的嵌入模型，便宜（每百万词约 0.02 美元）
- **S3 Vectors**：存向量的地方，按用量付费，没有一直开着的服务器。传统的向量数据库（比如 OpenSearch）最低每月要几百美元
- **Claude Haiku 4.5**：Anthropic 的快速、便宜的模型，回答问题够用

**为什么放在爱尔兰？** 写这一章时，我们以为免费计划不能"跨区域调用"，所以把知识库放在了本地就有 Claude、Titan 和 S3 Vectors 的爱尔兰（docs/plan.md 的决定 9）。后来实测：本区域的调用方式对这个账号不开放，而**欧盟跨区域推理配置**（模型 ID 以 `eu.` 开头，见第 7 步）可以用。两者都在欧盟内，就不再搬回法兰克福了。其余一切仍在法兰克福。资料都是公开的（UniProt 和你的论文），放在欧盟内的另一个区域没有问题。

### 公司里什么情况怎么做
- **先 RAG，后微调**：要让模型知道公司的资料，几乎总是先做 RAG
- 托管服务（Knowledge Bases）适合快速上线；需要精细控制时，公司也会自己写切段和检索。我们用托管的知识库做检索，生成这一步自己写，所以每一步都看得见

### 面试怎么说
> "RAG grounds a language model in our own documents: passages are embedded with Titan, stored in S3 Vectors through a Bedrock Knowledge Base, the closest ones are retrieved for each question, and Claude answers from those passages only, citing them. It keeps answers current and traceable without fine-tuning."

---

## 第 2 步 · 资料进货

### 在干什么
把两份资料准备好：UniProt 的人类蛋白注释，和你的 QAFI 论文。

### 背景
比喻：**把教科书和你的论文搬进图书馆**。
- **UniProt Swiss-Prot 人类蛋白**（`uniprot_sprot_human.dat.gz`，120 MB）：20,431 个经过专家审阅的人类蛋白，每个都有功能、细胞内位置、相关疾病、结构域，以及**已知突变**和**实验诱变**的记录。版本 2026_03，和我们的序列同一版。和其他公开数据一样，由 03b 号的"送货员" Lambda 从 UniProt 官方直接搬进 raw，并登记在 `config/sources.toml` 里
- **你的论文**：《Toward Advanced Quantitative Prediction of Missense Variant Effects》（IJMS 接收版）。知识库能直接读 Word 文件

### 执行

In [ ]:
sources = [s for s in ingest.load_sources(Path("config/sources.toml")) if s["key"].endswith("uniprot_sprot_human.dat.gz")]
receipt, = ingest.fetch_sources(frankfurt.client("lambda"), frankfurt.client("s3"), RAW, sources)
local = Path("data/knowledge/uniprot_sprot_human.dat.gz")
local.parent.mkdir(parents=True, exist_ok=True)
frankfurt.client("s3").download_file(RAW, receipt["key"], str(local))
print(f"本地：{local}（{local.stat().st_size / 1e6:.0f} MB）；论文：{PAPER.name}（{PAPER.stat().st_size / 1e6:.1f} MB）")

### 结果怎么看
- `accepted` 或 `skipped (already in raw)`：文件在 raw 里，校验通过
- 下载到你的 Mac 只是为了下一步整理，`data/` 文件夹不进 Git

---

## 第 3 步 · 整理成文档

### 在干什么
把 UniProt 的原始格式整理成每个蛋白一篇、结构清楚的文档，传到爱尔兰的存储桶。

### 背景
比喻：**每个蛋白一张卡片**。UniProt 的原始文件是给程序读的，像这样：

```
CC   -!- FUNCTION: Multifunctional transcription factor ... {ECO:0000269|PubMed:11025664,
CC       PubMed:12524540, ...}.
FT   VARIANT         175
FT                   /note="R -> H (in LFS; germline mutation ...)"
```

整理时做了三件事：
1. **按小标题组织**：功能、亚细胞定位、疾病、结构域、天然突变、实验诱变……。小标题让切出来的每一段都更好理解
2. **去掉噪音**：证据代码（`{ECO:…}`）和一长串文献编号，对读者和向量都没用
3. **突变写成我们的格式**：`R -> H` 在第 175 位，写成 `R175H`，和 API 用的写法一致

每篇文档旁边放一个**元数据**小文件（`P04637.md.metadata.json`）：蛋白编号、基因名、来源。检索时可以先按元数据筛选，比如"只在 TP53 的文档里找"。

### 执行

In [ ]:
started = time.time()
items = [(f"uniprot/{UNIPROT}/{accession}.md", text.encode(), metadata)
         for accession, text, metadata in knowledge.documents(local)]
items.append(("papers/ye2026_qafi.docx", PAPER.read_bytes(),
              {"collection": "paper", "title": "Toward Advanced Quantitative Prediction of Missense Variant Effects"}))
tp53 = next(body.decode() for key, body, _ in items if key.endswith("/P04637.md"))
print(f"{len(items) - 1:,} 篇蛋白文档，共 {sum(len(b) for _, b, _ in items[:-1]) / 1e6:.0f} MB 文字，用了 {time.time() - started:.0f} 秒")
print(tp53[:900], "\n...\n", tp53[tp53.find("- R175H"):][:260])

### 结果怎么看
- 20,431 篇蛋白文档，约 47 MB 文字（约 1,200 万个词）
- TP53 的文档开头：名字、编号、长度，然后是功能。中间省略
- 最后几行是 R175H 的记录：在 Li-Fraumeni 综合征（LFS）中作为胚系突变出现，也见于散发癌症

文档先放在内存里，第 5 步传到爱尔兰。

---

## 第 4 步 · 建知识库

### 在干什么
用 Terraform 在爱尔兰建知识库需要的一切。

### 背景
比喻：**按"意思"排架的图书馆**。要建的东西（`infra/dev/knowledge.tf`）：

| 资源 | 作用 |
|---|---|
| S3 存储桶 `…-knowledge` | 放文档（就是上一步的那些卡片） |
| S3 Vectors 向量桶 + 索引 `passages` | 放每一段的向量，1,024 维，按方向相近（余弦）比较 |
| IAM 角色 `qafi-dev-knowledge` | 知识库的身份：只能读这个桶、调用 Titan、读写这个索引 |
| 知识库 `qafi-dev-knowledge` | 把以上连起来 |
| 两个数据源 `uniprot`、`papers` | 告诉知识库去哪里读文档，以及**怎么切段** |

**两种资料，两种切法**：
- **蛋白文档**：格式整齐，**每 300 个词切一段，前后重叠 20%**，免得一句话被从中间切断
- **论文**：段落长短不一，按**意思的转折**来切（semantic chunking）：相邻句子意思变化最大的地方下刀

**加密**：这些都是公开资料，用 S3 自带的加密就够了。平台的密钥（01 号建的）留给我们自己的数据，而且密钥是按区域收费的，再建一把每月多 1 美元。

### 执行

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color 2>&1 | tail -1")
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

`Plan: 10 to add, 0 to change, 0 to destroy`。确认后执行：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")
KNOWLEDGE = terraform_output("knowledge")
KNOWLEDGE

### 去网页上看
AWS 控制台，右上角切到 **Europe (Ireland)** → Amazon Bedrock → **Knowledge Bases** → `qafi-dev-knowledge`：能看到嵌入模型、向量存储（S3 Vectors）和两个数据源。

### 面试怎么说
> "The knowledge base is Terraform: an S3 bucket for documents, an S3 Vectors index, a scoped IAM role and two data sources with different chunking: fixed-size with overlap for uniform protein entries, semantic chunking for the paper."

---

## 第 5 步 · 上传和导入

### 在干什么
把文档传到爱尔兰的存储桶，再让知识库读所有文档、切段、算向量、存进索引。

### 背景
比喻：**把卡片切成小段、编好索引、上架**。这一步叫**导入**（ingestion）。对每篇文档：切段 → 每段交给 Titan 变成 1,024 个数 → 连同原文和元数据存进 S3 Vectors。

**增量**：以后 UniProt 出了新版，重新上传、再导入一次，没变的文档会被跳过，只处理新的和改过的。

### 执行
先上传。第一次约 3 分钟（4 万个文件：每篇文档加一个元数据文件）；以后再运行，**内容没变的文件会跳过**（比对文件内容的 MD5 和 S3 存的指纹 ETag），几秒钟就完，下一次导入也只处理变了的文档：

In [ ]:
started = time.time()
print(f"{knowledge.upload(ireland.client('s3'), KNOWLEDGE['bucket'], items):,} 个文件有变化、已上传，用了 {time.time() - started:.0f} 秒")

再导入：先是论文（半分钟），再是 2 万篇蛋白文档（约 100 分钟）：

In [ ]:
agent = ireland.client("bedrock-agent")
for name in ["papers", "uniprot"]:
    stats = knowledge.ingest(agent, KNOWLEDGE["knowledge_base"], KNOWLEDGE["data_sources"][name])
    print(f"{name}: 扫描 {stats['numberOfDocumentsScanned']:,} 篇，新索引 {stats['numberOfNewDocumentsIndexed']:,}，"
          f"更新 {stats['numberOfModifiedDocumentsIndexed']:,}，失败 {stats['numberOfDocumentsFailed']:,}，用了 {stats['minutes']} 分钟")

### 结果怎么看
写这一章时的结果：
- **papers**：1 篇，半分钟
- **uniprot**：扫描 20,431 篇，新索引 20,431 篇，失败 0 篇，约 96 分钟
- 知识库里一共 **58,248 段**：大多数蛋白两三段，TP53 这样有上千条突变记录的大文档有几十段

再运行一次时，没有变化的文档会被跳过（`新索引 0`），几分钟就完。

**写这一章时踩过的两个坑**（已经修好，留在这里是因为公司里也常见）：
- 7,155 个蛋白的基因名后面粘着证据代码，比如 `CFTR {ECO:0000312|HGNC:HGNC:1884}`，这样"按基因名筛选"就找不到它们。TP53 恰好没有，所以只看 TP53 发现不了：**检查数据要看整体，不能只看一个例子**
- 148 个蛋白没有基因名，元数据里写了空字符串，知识库直接拒收这些文档。现在没有基因名就不写这个字段

---

## 第 6 步 · 检索

### 在干什么
只做检索，不生成：看看知识库按意思能找到什么。

### 背景
四个问题：
1. **不提名字，用比喻**："哪个蛋白守护基因组，在 DNA 损伤后让细胞停止分裂或凋亡？"
2. **不提名字，用 UniProt 的说法**："哪个转录因子是抑癌蛋白，在 DNA 损伤后诱导细胞周期停滞或凋亡？"
3. **先筛选再找**：只在 TP53 的文档里找 "R175H"
4. **问论文**：QAFI 是怎么验证的

每个结果有一个**分数**（0 到 1，越高越相近）和**出处**（哪篇文档）。

### 执行

In [ ]:
runtime = ireland.client("bedrock-agent-runtime")
KB = KNOWLEDGE["knowledge_base"]
def show(passages):
    return pd.DataFrame([{"分数": p["score"], "出处": p["source"], "内容": p["text"][:160].replace("\n", " ")} for p in passages])
for question in ["Which protein guards the genome and makes cells stop dividing or die after DNA damage?",
                 "Which transcription factor acts as a tumor suppressor and induces cell cycle arrest or apoptosis after DNA damage?"]:
    started = time.time()
    sources = [p["source"] for p in knowledge.retrieve(runtime, KB, question, k=10)]
    rank = sources.index("P04637.md") + 1 if "P04637.md" in sources else None
    print(f"{question[:60]}…  TP53 排第 {rank} 名" if rank else f"{question[:60]}…  前 10 名里没有 TP53", f"（{time.time() - started:.1f} 秒）")
display(show(knowledge.retrieve(runtime, KB, "R175H", k=3, where={"equals": {"key": "gene_name", "value": "TP53"}})))
show(knowledge.retrieve(runtime, KB, "How was QAFI validated?", k=3, where={"equals": {"key": "collection", "value": "paper"}}))

### 结果怎么看
写这一章时的结果：
- **用比喻问**（"守护基因组"）：前 10 名里**没有** TP53，找到的是其他和 DNA 损伤有关的蛋白
- **用 UniProt 的说法问**（"转录因子、抑癌、细胞周期停滞或凋亡"）：**TP53 排第 1**
- **只在 TP53 里找 R175H**：排第 1 的那一段就包含 `R175H: in LFS; germline mutation…` 这一行
- **问论文**：三段都来自论文，讲的正是外部验证（包括 CAGI 7）
- 一次检索不到 1 秒

**最重要的一课**：按意思检索并不神奇，它找的是**和文档写法相近**的段落。用户的说法和资料的写法差得远（比喻、口语），就可能找不到。公司里的对策：让模型先把问题改写成专业说法再去检索（query rewriting），或者把关键词检索和向量检索结合起来（hybrid search）

### 公司里什么情况怎么做
- 检索的质量决定了回答的上限：找错了段，模型再好也答不对。所以先单独看检索，再看生成
- **元数据筛选**很常用：按部门、按日期、按产品先缩小范围，再按意思找

### 面试怎么说
> "Retrieval is evaluated on its own first. Semantic search found TP53 from a description in UniProt's own terms, but not from a metaphor like 'guards the genome': it matches how the documents are written. Metadata filters narrow the search to one protein before the vector search."

---

## 第 7 步 · 带出处地回答

### 在干什么
把检索到的段落交给 Claude，让它带着出处回答。

### 背景
比喻：**开卷答题，每句注明出处**。给 Claude 的指令（`knowledge.SYSTEM`）只有几条：
- 只用给你的编号段落回答
- 每句话后面标明出处 [1] [2]
- 段落里没有答案，就说在知识库里找不到，**不要用你自己的知识**

三个问题：一个蛋白问题、一个论文问题、一个**知识库里根本没有答案**的问题，看它会不会老实说不知道。

**怎么调用 Claude**：用 Bedrock 的标准接口 `bedrock-runtime`，模型 ID 是 `eu.anthropic.claude-haiku-4-5-…`。前缀 `eu.` 表示**欧盟跨区域推理配置**：Bedrock 把每个请求送到欧盟内当时有空闲的区域，**数据不出欧盟**，价格比单个区域贵约 10%。请求和回答用 Anthropic 的 Messages 格式；boto3 用你的 AWS 身份签名（和 10 号调用 API 一样，不需要额外的密钥），被限流时自动重试（`qafi_platform/claude.py`）。

### 执行
先确认你的账号已经能用 Claude（开始前那一步：用例说明 + 使用协议）：

In [ ]:
available = ireland.client("bedrock").get_foundation_model_availability(modelId="anthropic.claude-haiku-4-5-20251001-v1:0")
print("Claude Haiku 4.5：", available["agreementAvailability"]["status"])

`AVAILABLE` 就可以往下走。如果是 `NOT_AVAILABLE`，回到开头提交用例说明、接受使用协议，几分钟后再运行这一格。

In [ ]:
questions = [
    ("What is known about the TP53 variant R175H?", {"equals": {"key": "gene_name", "value": "TP53"}}),
    ("How was QAFI validated on proteins that were not used to build it?", {"equals": {"key": "collection", "value": "paper"}}),
    ("Which drug did the FDA approve in 2027 for TP53-mutant cancers?", None),
]
for question, where in questions:
    passages = knowledge.retrieve(runtime, KB, question, k=5, where=where)
    started = time.time()
    result = knowledge.answer(ireland, question, passages)
    print(f"问：{question}\n答：{result['answer']}\n引用：{result['cited']}   "
          f"（{time.time() - started:.1f} 秒，{result['input_tokens']:,} + {result['output_tokens']} 个词，{result['usd'] * 100:.2f} 美分）\n")

### 结果怎么看
{answer_text}

### 公司里什么情况怎么做
- **让模型可以说"不知道"**，比让它什么都答更重要。系统提示里明确写出来，并且专门测试
- 每个回答都记下用了哪些段落、花了多少钱：出了问题能追查，费用能预估

### 面试怎么说
> "Generation is a separate, inspectable step: Claude Haiku 4.5 on Bedrock gets numbered passages and must cite them or say the answer is not in the knowledge base. Each answer records its sources, tokens and cost."

---

## 第 8 步 · 评估

### 在干什么
出一套有标准答案的题，给 RAG 打分。

### 背景
比喻：**出一套有标准答案的题**。08、09 号我们一直强调"考卷"，RAG 也一样。不打分，就不知道改了切段方式、换了模型之后是变好还是变坏。

每道题有两个检查：
- **检索对不对**：应该用到的那篇文档，是否在找到的前 5 段里（hit@5）
- **回答对不对**：回答里有没有关键事实（一个关键词）。最后一题知识库里没有答案，正确的回答是"找不到"

（公司里还会用另一个模型来给回答打分，叫 LLM-as-a-judge。我们先用最简单、最透明的关键词检查。）

### 执行

In [ ]:
exam = [  # question, the document that holds the answer, a word the answer must contain
    ("Which protein is a tumor suppressor that induces cell cycle arrest or apoptosis after DNA damage?", "P04637.md", "TP53"),
    ("Mutations in which gene cause cystic fibrosis?", "P13569.md", "CFTR"),
    ("Which disease is caused by expansion of a polyglutamine stretch in huntingtin?", "P42858.md", "Huntington"),
    ("What does hemoglobin subunit beta carry?", "P68871.md", "oxygen"),
    ("Where in the cell is lamin A found?", "P02545.md", "nucle"),
    ("Which syndrome is caused by mutations in FBN1?", "P35555.md", "Marfan"),
    ("What is the role of BRCA1 in DNA repair?", "P38398.md", "homologous recombination"),
    ("What does the PTEN mutation C124S do in experiments?", "P60484.md", "phosphatase"),
    ("Which proteins were used in the CAGI 7 benchmark of QAFI?", "ye2026_qafi.docx", "ARSA"),
    ("What is QAFIsplit?", "ye2026_qafi.docx", "QAFI"),
    ("Who won the 2026 football World Cup?", None, "cannot find"),
]
rows = []
for question, expected, word in exam:
    passages = knowledge.retrieve(runtime, KB, question, k=5)
    result = knowledge.answer(ireland, question, passages)
    rows.append({"问题": question[:60], "检索对": expected is None or expected in [p["source"] for p in passages],
                 "回答对": word.lower() in result["answer"].lower(), "引用": ", ".join(result["cited"])[:40], "美分": result["usd"] * 100})
scores = pd.DataFrame(rows)
print(f"检索对 {scores['检索对'].sum()}/{len(scores)}，回答对 {scores['回答对'].sum()}/{len(scores)}，"
      f"一共 {scores['美分'].sum():.1f} 美分")
scores

### 结果怎么看
{eval_text}

### 公司里什么情况怎么做
- RAG 上线前一定有一套**评估题**，每次改动（换模型、改切段、改提示词）都重跑一遍，和上线前的分数比
- 答错的题要逐个看：是**检索**没找到，还是找到了但**生成**没用好？两种问题的修法完全不同

### 面试怎么说
> "A small labelled exam scores retrieval (hit@5) and answers (key fact present, and 'cannot find' when the knowledge base has no answer). Failures are split into retrieval misses and generation misses, which need different fixes."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 11 RAG knowledge base: UniProt and the QAFI paper, Claude with citations") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 原理 | 向量、检索、生成 | 开卷考试：先找资料再回答 |
| 2 进货 | UniProt 注释进 raw；你的论文 | 资料和别的数据一样，从官方来、版本写死 |
| 3 整理 | 每个蛋白一篇 Markdown + 元数据 | 结构清楚、去掉噪音，检索才准 |
| 4 知识库 | 爱尔兰：S3 + S3 Vectors + 知识库，两种切法 | 按资料的样子选切法 |
| 5 导入 | 切段、嵌入、存索引 | 增量：没变的跳过 |
| 6 检索 | 不提名字也能按意思找到；元数据筛选 | 检索决定回答的上限 |
| 7 回答 | Claude 只用编号段落，标出处，可以说不知道 | 每个回答可追溯、可计价 |
| 8 评估 | 11 道题：检索对几道、回答对几道 | 改了什么都要重新考 |

**一句话**：我们有了一个"开卷"的蛋白问答：每句话都有出处，资料里没有的就说不知道。

## 面试题

1. **What is RAG and why not fine-tune?**
   > "Retrieve the relevant passages from your own documents, then generate from them with citations. Documents change and must be traceable; RAG updates by re-ingesting and cites every statement, fine-tuning does neither."

2. **How did you design chunking?**
   > "By document type: fixed 300-token chunks with 20% overlap for uniform UniProt entries, semantic chunking for the paper. Documents were cleaned of evidence codes and reference lists before embedding."

3. **How do you stop hallucinations?**
   > "The model must answer only from numbered passages, cite them, and say when the answer is not there; the exam includes an unanswerable question to test exactly that."

4. **How do you evaluate a RAG system?**
   > "Separately for retrieval (hit@k of the expected source) and generation (key facts, refusal when appropriate), on a fixed exam rerun after every change; LLM-as-a-judge for open-ended answers."

**下一步 12 · Agent**：让 Claude 自己决定用什么工具：查 10 号 API 拿分数、查这个知识库拿背景，再把两者结合起来回答"TP53 R175H 有害吗，为什么"。